In [ ]:
# | default_exp preprocessing.ocr.dashscope

In [ ]:
%load_ext autoreload
%autoreload 2

# DashScope Qwen document parsing

> Recursively parse PDF pages and PNG/JPEG images into QwenVL Markdown with Alibaba Cloud DashScope.

This notebook is a dedicated alternative to `03.preprocessing.ocr.glm.ipynb`. It sends each complete page through DashScope's OpenAI-compatible multimodal interface using the exact `qwenvl markdown` document-parsing prompt. Thinking is disabled and high-resolution visual processing is enabled by default. Model output is retained verbatim so QwenVL table, formula, image-position, and layout markup is not lost.

Each source produces a UTF-8 Markdown document under `.md_dashscope/` and a sibling `.qwen.json` metadata file. Relative directories and page order are preserved. Files and PDF pages run concurrently behind one global request limit, while each completed output pair is published atomically with Markdown last.

The notebook loads `DASHSCOPE_API_KEY`, `DASHSCOPE_API_URL`, optional `OPENAILIKED_OCR_MODEL`, `OPENAILIKED_MAX_CONCURRENCY`, and `OPENAILIKED_PAGE_CONCURRENCY` from the project-root `.env` without overriding process variables. The real batch call at the end is opt-in and is excluded from tests.

In [ ]:
# | export
import asyncio
import base64
import json
import os
from collections import Counter
from dataclasses import dataclass
from html import escape
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter
from typing import Any, Callable, Literal

import pymupdf
from dotenv import load_dotenv
from openai import AsyncOpenAI
from PIL import Image
from tqdm.auto import tqdm


In [ ]:
# | export
def _find_project_root() -> Path:
    """Find the nearest parent containing pyproject.toml."""
    starts: list[Path] = []
    module_file = globals().get("__file__")
    if isinstance(module_file, str):
        starts.append(Path(module_file).resolve().parent)
    starts.append(Path.cwd().resolve())
    for start in starts:
        for candidate in [start, *start.parents]:
            if (candidate / "pyproject.toml").is_file():
                return candidate
    return Path.cwd().resolve()


PROJ_ROOT = _find_project_root()
load_dotenv(PROJ_ROOT / ".env", override=False)

In [ ]:
# | export
@dataclass(frozen=True)
class DashScopeOCRResult:
    """Outcome of parsing one PDF or image with DashScope Qwen."""

    source_path: Path
    markdown_path: Path
    metadata_path: Path
    status: Literal["processed", "skipped", "failed"]
    pages_total: int = 0
    pages_completed: int = 0
    elapsed_s: float = 0.0
    error: str | None = None


_SUPPORTED_SUFFIXES = frozenset({".pdf", ".png", ".jpg", ".jpeg"})
_IMAGE_SUFFIXES = frozenset({".png", ".jpg", ".jpeg"})
_DEFAULT_MODEL = "qwen3.7-plus"
_DEFAULT_BASE_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1"
_DEFAULT_PROMPT = "qwenvl markdown"
_MAX_DATA_URL_BYTES = 10 * 1024 * 1024

In [ ]:
# | export
def _resolve_root(root_folder: Path | str) -> Path:
    root = Path(root_folder).expanduser().resolve()
    if not root.exists():
        raise FileNotFoundError(f"OCR root does not exist: {root}")
    if not root.is_dir():
        raise NotADirectoryError(f"OCR root is not a directory: {root}")
    return root


def _resolve_output_dir_name(output_dir_name: str) -> str:
    normalized = output_dir_name.strip()
    path = Path(normalized)
    if (
        not normalized
        or path.is_absolute()
        or len(path.parts) != 1
        or normalized in {".", ".."}
    ):
        raise ValueError("output_dir_name must be one relative directory name")
    return normalized


def _metadata_path(markdown_path: Path) -> Path:
    return markdown_path.with_suffix(".qwen.json")


def _ocr_jobs(
    root: Path,
    output_dir_name: str,
) -> list[tuple[Path, Path, Path]]:
    """Return deterministic source/Markdown/metadata triples."""
    output_root = root / _resolve_output_dir_name(output_dir_name)
    sources = [
        path
        for path in root.rglob("*")
        if path.is_file()
        and path.suffix.casefold() in _SUPPORTED_SUFFIXES
        and not path.is_relative_to(output_root)
    ]
    sources.sort(
        key=lambda path: (
            path.relative_to(root).as_posix().casefold(),
            path.relative_to(root).as_posix(),
        )
    )

    jobs: list[tuple[Path, Path, Path]] = []
    targets: dict[str, Path] = {}
    for source_path in sources:
        relative = source_path.relative_to(root).with_suffix(".md")
        markdown_path = output_root / relative
        collision_key = markdown_path.as_posix().casefold()
        if previous := targets.get(collision_key):
            raise ValueError(
                f"OCR output collision: {previous} and {source_path} both map to "
                f"{markdown_path}"
            )
        targets[collision_key] = source_path
        jobs.append((source_path, markdown_path, _metadata_path(markdown_path)))
    return jobs

In [ ]:
# | export
def _resolve_model(model: str | None) -> str:
    if model is not None:
        if not model.strip():
            raise ValueError("model must not be empty")
        return model.strip()
    return os.getenv("OPENAILIKED_OCR_MODEL", "").strip() or _DEFAULT_MODEL


def _resolve_prompt(prompt: str) -> str:
    if not prompt.strip():
        raise ValueError("prompt must not be empty")
    return prompt.strip()


def _positive_int(value: int | None, env_name: str, default: int) -> int:
    if value is None:
        configured = os.getenv(env_name, "").strip()
        if configured:
            try:
                value = int(configured)
            except ValueError as error:
                raise ValueError(f"{env_name} must be a positive integer") from error
        else:
            value = default
    if value <= 0:
        raise ValueError(f"{env_name} must be a positive integer")
    return value


def _resolve_base_url(base_url: str | None) -> str:
    resolved = (
        base_url.strip()
        if base_url is not None
        else os.getenv("DASHSCOPE_API_URL", "").strip() or _DEFAULT_BASE_URL
    )
    if (
        not resolved.startswith(("http://", "https://"))
        or "/compatible-mode/" not in resolved
    ):
        raise RuntimeError(
            "DASHSCOPE_API_URL must be an OpenAI-compatible HTTP(S) endpoint"
        )
    return resolved


def _create_client(base_url: str | None, request_timeout_s: float) -> AsyncOpenAI:
    api_key = os.getenv("DASHSCOPE_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError(
            f"DASHSCOPE_API_KEY is not configured in {PROJ_ROOT / '.env'}"
        )
    return AsyncOpenAI(
        api_key=api_key,
        base_url=_resolve_base_url(base_url),
        timeout=request_timeout_s,
    )

In [ ]:
# | export
def _load_image_pixmap(path: Path) -> pymupdf.Pixmap:
    """Load a PNG/JPEG at native resolution and composite transparency on white."""
    try:
        with Image.open(path) as image:
            image.load()
            rgba = image.convert("RGBA")
            background = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            rgb = Image.alpha_composite(background, rgba).convert("RGB")
            return pymupdf.Pixmap(
                pymupdf.csRGB,
                rgb.width,
                rgb.height,
                rgb.tobytes(),
                False,
            )
    except Exception as error:
        raise ValueError(f"Could not decode image {path}: {error}") from error


def _data_url(image_bytes: bytes, mime_type: str) -> str:
    return f"data:{mime_type};base64,{base64.b64encode(image_bytes).decode('ascii')}"


def _image_data_url(pixmap: pymupdf.Pixmap) -> tuple[str, str]:
    png_url = _data_url(pixmap.tobytes("png"), "image/png")
    if len(png_url.encode("ascii")) <= _MAX_DATA_URL_BYTES:
        return png_url, "image/png"

    jpeg_url = _data_url(
        pixmap.tobytes("jpeg", jpg_quality=90),
        "image/jpeg",
    )
    if len(jpeg_url.encode("ascii")) <= _MAX_DATA_URL_BYTES:
        return jpeg_url, "image/jpeg"
    raise ValueError(
        "Image payload exceeds DashScope's 10 MiB Base64 input limit; "
        "reduce PDF dpi or the source image dimensions"
    )

In [ ]:
# | export
def _value(obj: object, name: str, default: Any = None) -> Any:
    if isinstance(obj, dict):
        return obj.get(name, default)
    return getattr(obj, name, default)


def _response_record(response: object) -> tuple[str, dict[str, Any]]:
    choices = _value(response, "choices")
    if not isinstance(choices, (list, tuple)) or not choices:
        raise ValueError("DashScope document parsing returned no choices")
    choice = choices[0]
    message = _value(choice, "message")
    content = _value(message, "content")
    if not isinstance(content, str) or not content.strip():
        raise ValueError("DashScope document parsing returned an empty response")

    usage_obj = _value(response, "usage")
    usage: dict[str, int] = {}
    if usage_obj is not None:
        for name in ("prompt_tokens", "completion_tokens", "total_tokens"):
            token_count = _value(usage_obj, name)
            if isinstance(token_count, int):
                usage[name] = token_count

    return content, {
        "response_id": _value(response, "id"),
        "finish_reason": _value(choice, "finish_reason"),
        "usage": usage,
    }


async def _parse_pixmap(
    pixmap: pymupdf.Pixmap,
    page_number: int,
    *,
    client: AsyncOpenAI,
    model: str,
    prompt: str,
    high_resolution_images: bool,
    request_semaphore: asyncio.Semaphore | None,
) -> tuple[str, dict[str, Any]]:
    data_url, mime_type = _image_data_url(pixmap)
    started_at = perf_counter()

    async def send_request() -> object:
        return await client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "image_url", "image_url": {"url": data_url}},
                        {"type": "text", "text": prompt},
                    ],
                }
            ],
            temperature=0,
            extra_body={
                "enable_thinking": False,
                "vl_high_resolution_images": high_resolution_images,
            },
        )

    if request_semaphore is None:
        response = await send_request()
    else:
        async with request_semaphore:
            response = await send_request()
    content, response_metadata = _response_record(response)
    return content, {
        "page_number": page_number,
        "width": pixmap.width,
        "height": pixmap.height,
        "input_mime_type": mime_type,
        "elapsed_s": perf_counter() - started_at,
        "content": content,
        **response_metadata,
    }

In [ ]:
# | export
def _publish_output_pair(
    markdown_path: Path,
    markdown: str,
    metadata: dict[str, Any],
) -> None:
    """Publish metadata first and expose Markdown last, restoring old outputs on error."""
    metadata_path = _metadata_path(markdown_path)
    markdown_path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(
        dir=markdown_path.parent,
        prefix=f".{markdown_path.stem}.qwen-",
    ) as temporary_directory:
        stage_root = Path(temporary_directory)
        stage_markdown = stage_root / markdown_path.name
        stage_metadata = stage_root / metadata_path.name
        stage_markdown.write_text(markdown, encoding="utf-8", newline="\n")
        stage_metadata.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2) + "\n",
            encoding="utf-8",
            newline="\n",
        )

        publications = [
            (stage_metadata, metadata_path),
            (stage_markdown, markdown_path),
        ]
        backups: list[tuple[Path, Path]] = []
        published: list[tuple[Path, Path]] = []
        try:
            for index, (_, final_path) in enumerate(publications):
                if final_path.exists():
                    backup_path = stage_root / f"backup-{index}"
                    final_path.replace(backup_path)
                    backups.append((backup_path, final_path))
            for staged_path, final_path in publications:
                staged_path.replace(final_path)
                published.append((final_path, staged_path))
        except Exception:
            for final_path, staged_path in reversed(published):
                if final_path.exists():
                    final_path.replace(staged_path)
            for backup_path, final_path in reversed(backups):
                if backup_path.exists():
                    backup_path.replace(final_path)
            raise


def _target_state(
    source: Path,
    target: Path,
    *,
    overwrite: bool,
) -> DashScopeOCRResult | None:
    metadata_path = _metadata_path(target)
    if target.exists() and not target.is_file():
        return DashScopeOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            error="Markdown target is not a file",
        )
    if metadata_path.exists() and not metadata_path.is_file():
        return DashScopeOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            error="Metadata target is not a file",
        )
    if target.is_file() and not overwrite:
        return DashScopeOCRResult(source, target, metadata_path, "skipped")
    return None


def _assembled_markdown(page_contents: list[str]) -> str:
    sections = [
        f"<!-- Page {page_number} -->\n\n{content}"
        for page_number, content in enumerate(page_contents, start=1)
    ]
    markdown = "\n\n".join(sections)
    return markdown if markdown.endswith("\n") else markdown + "\n"

In [ ]:
# | export
async def ocr_pdf(
    pdf_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 200,
    overwrite: bool = False,
    high_resolution_images: bool = True,
    page_concurrency: int = 2,
    request_semaphore: asyncio.Semaphore | None = None,
    page_started: Callable[[int], None] | None = None,
    page_progress: Callable[[int, int, float], None] | None = None,
) -> DashScopeOCRResult:
    """Parse one PDF into QwenVL Markdown with concurrent ordered pages."""
    source = Path(pdf_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = _metadata_path(target)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if page_concurrency <= 0:
        raise ValueError("page_concurrency must be greater than zero")
    if state := _target_state(source, target, overwrite=overwrite):
        return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    tasks: list[asyncio.Task[tuple[int, str, dict[str, Any]]]] = []
    try:
        if not source.is_file():
            raise FileNotFoundError(f"PDF does not exist: {source}")
        page_gate = asyncio.Semaphore(page_concurrency)
        with pymupdf.open(source) as document:
            if document.needs_pass:
                raise ValueError("PDF requires a password")
            pages_total = document.page_count
            if pages_total == 0:
                raise ValueError("PDF contains no pages")
            if page_started is not None:
                page_started(pages_total)

            async def process_page(
                page_number: int,
            ) -> tuple[int, str, dict[str, Any]]:
                nonlocal pages_completed
                page_started_at = perf_counter()
                async with page_gate:
                    page = document.load_page(page_number - 1)
                    pixmap = page.get_pixmap(dpi=dpi, alpha=False)
                    content, record = await _parse_pixmap(
                        pixmap,
                        page_number,
                        client=client,
                        model=selected_model,
                        prompt=selected_prompt,
                        high_resolution_images=high_resolution_images,
                        request_semaphore=request_semaphore,
                    )
                elapsed_s = perf_counter() - page_started_at
                record["elapsed_s"] = elapsed_s
                pages_completed += 1
                if page_progress is not None:
                    page_progress(pages_completed, pages_total, elapsed_s)
                return page_number, content, record

            tasks = [
                asyncio.create_task(process_page(page_number))
                for page_number in range(1, pages_total + 1)
            ]
            try:
                parsed_pages = await asyncio.gather(*tasks)
            finally:
                for task in tasks:
                    if not task.done():
                        task.cancel()
                await asyncio.gather(*tasks, return_exceptions=True)

        parsed_pages.sort(key=lambda item: item[0])
        page_contents = [content for _, content, _ in parsed_pages]
        page_records = [record for _, _, record in parsed_pages]
        document_elapsed_s = perf_counter() - started_at
        _publish_output_pair(
            target,
            _assembled_markdown(page_contents),
            {
                "schema_version": 1,
                "source": str(source),
                "model": selected_model,
                "prompt": selected_prompt,
                "dpi": dpi,
                "high_resolution_images": high_resolution_images,
                "elapsed_s": document_elapsed_s,
                "pages": page_records,
            },
        )
        return DashScopeOCRResult(
            source,
            target,
            metadata_path,
            "processed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
        )
    except Exception as error:
        return DashScopeOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
async def ocr_image(
    image_path: Path | str,
    markdown_path: Path | str,
    *,
    client: AsyncOpenAI,
    model: str | None = None,
    prompt: str = _DEFAULT_PROMPT,
    overwrite: bool = False,
    high_resolution_images: bool = True,
    request_semaphore: asyncio.Semaphore | None = None,
) -> DashScopeOCRResult:
    """Parse one PNG/JPEG image into QwenVL Markdown at native resolution."""
    source = Path(image_path).expanduser().resolve()
    target = Path(markdown_path).expanduser().resolve()
    metadata_path = _metadata_path(target)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if state := _target_state(source, target, overwrite=overwrite):
        return state

    started_at = perf_counter()
    pages_total = 0
    pages_completed = 0
    try:
        if not source.is_file():
            raise FileNotFoundError(f"Image does not exist: {source}")
        if source.suffix.casefold() not in _IMAGE_SUFFIXES:
            raise ValueError(f"Unsupported image type: {source.suffix or '<none>'}")
        pixmap = _load_image_pixmap(source)
        pages_total = 1
        content, page_record = await _parse_pixmap(
            pixmap,
            1,
            client=client,
            model=selected_model,
            prompt=selected_prompt,
            high_resolution_images=high_resolution_images,
            request_semaphore=request_semaphore,
        )
        pages_completed = 1
        document_elapsed_s = perf_counter() - started_at
        _publish_output_pair(
            target,
            _assembled_markdown([content]),
            {
                "schema_version": 1,
                "source": str(source),
                "model": selected_model,
                "prompt": selected_prompt,
                "dpi": None,
                "high_resolution_images": high_resolution_images,
                "elapsed_s": document_elapsed_s,
                "pages": [page_record],
            },
        )
        return DashScopeOCRResult(
            source,
            target,
            metadata_path,
            "processed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
        )
    except Exception as error:
        return DashScopeOCRResult(
            source,
            target,
            metadata_path,
            "failed",
            pages_total,
            pages_completed,
            perf_counter() - started_at,
            f"{type(error).__name__}: {error}",
        )

In [ ]:
# | export
@dataclass
class _ActivePageProgress:
    source_path: Path
    pages_total: int | None = None
    pages_completed: int = 0
    last_page_elapsed_s: float | None = None


def _running_in_notebook() -> bool:
    try:
        from IPython import get_ipython
    except ImportError:
        return False
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


class _OCRFolderProgress:
    """Render fixed page and file progress in terminals and notebooks."""

    def __init__(self, total_files: int, description: str, show_pages: bool) -> None:
        self._total_files = total_files
        self._files_completed = 0
        self._description = description
        self._show_pages = show_pages
        self._active_order: list[int] = []
        self._active: dict[int, _ActivePageProgress] = {}
        self._displayed_index: int | None = None
        self._notebook_mode = _running_in_notebook()
        self._html_factory: Callable[[str], Any] | None = None
        self._display: Callable[..., Any] | None = None
        self._display_handle: Any = None
        self._page_bar: Any = None
        self._file_bar: Any = None

        if self._notebook_mode:
            from IPython.display import HTML, display

            self._html_factory = HTML
            self._display = display
            self._refresh_notebook()
        else:
            self._page_bar = (
                tqdm(
                    total=None,
                    desc="Qwen pages: waiting",
                    unit="page",
                    position=0,
                    leave=False,
                    dynamic_ncols=True,
                )
                if show_pages
                else None
            )
            self._file_bar = tqdm(
                total=total_files,
                desc=description,
                unit="file",
                position=1 if show_pages else 0,
                leave=True,
                dynamic_ncols=True,
            )

    def start_file(
        self,
        index: int,
        source_path: Path,
        pages_total: int | None = None,
    ) -> None:
        self._active[index] = _ActivePageProgress(source_path, pages_total)
        self._active_order.append(index)
        self._refresh_page_bar()

    def set_pages_total(self, index: int, pages_total: int) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            self._refresh_page_bar()

    def complete_page(
        self,
        index: int,
        pages_completed: int,
        pages_total: int,
        elapsed_s: float,
    ) -> None:
        state = self._active.get(index)
        if state is not None:
            state.pages_total = pages_total
            state.pages_completed = pages_completed
            state.last_page_elapsed_s = elapsed_s
            self._refresh_page_bar()

    def finish_file(self, index: int) -> None:
        self._active.pop(index, None)
        if index in self._active_order:
            self._active_order.remove(index)
        self._files_completed += 1
        if self._file_bar is not None:
            self._file_bar.update(1)
        self._refresh_page_bar()

    def _refresh_page_bar(self) -> None:
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is None:
            return
        if not self._active_order:
            self._displayed_index = None
            self._page_bar.clear()
            self._page_bar.set_description_str("Qwen pages: waiting", refresh=False)
            self._page_bar.total = None
            self._page_bar.n = 0
            self._page_bar.set_postfix_str("", refresh=False)
            self._page_bar.refresh()
            return

        newest_index = self._active_order[-1]
        state = self._active[newest_index]
        if newest_index != self._displayed_index:
            self._page_bar.clear()
            self._displayed_index = newest_index
            self._page_bar.n = 0
            self._page_bar.last_print_n = 0
            now = self._page_bar._time()
            self._page_bar.start_t = now
            self._page_bar.last_print_t = now
        self._page_bar.set_description_str(
            f"Qwen pages: {state.source_path}",
            refresh=False,
        )
        self._page_bar.total = state.pages_total
        self._page_bar.n = state.pages_completed
        elapsed = state.last_page_elapsed_s
        self._page_bar.set_postfix_str(
            f"last={elapsed:.2f}s" if elapsed is not None else "",
            refresh=False,
        )
        self._page_bar.refresh()

    @staticmethod
    def _html_progress_row(
        label: str,
        completed: int,
        total: int | None,
        detail: str = "",
    ) -> str:
        count = f"{completed}/{total}" if total is not None else f"{completed}/?"
        percentage = 100 * completed / total if total else 0
        progress = (
            f'<progress value="{min(completed, total)}" max="{total}" '
            'style="width:100%;height:0.8rem"></progress>'
            if total is not None
            else '<progress style="width:100%;height:0.8rem"></progress>'
        )
        suffix = f" &nbsp; {escape(detail)}" if detail else ""
        return (
            '<div style="margin:0 0 0.45rem 0">'
            '<div style="display:flex;gap:1rem;justify-content:space-between;">'
            f'<span style="overflow-wrap:anywhere">{escape(label)}</span>'
            f'<span style="white-space:nowrap">{percentage:.0f}% &nbsp; '
            f"{count}{suffix}</span></div>{progress}</div>"
        )

    def _notebook_html(self) -> str:
        rows: list[str] = []
        if self._show_pages:
            if self._active_order:
                state = self._active[self._active_order[-1]]
                elapsed = state.last_page_elapsed_s
                detail = f"last={elapsed:.2f}s" if elapsed is not None else ""
                rows.append(
                    self._html_progress_row(
                        f"Qwen pages: {state.source_path}",
                        state.pages_completed,
                        state.pages_total,
                        detail,
                    )
                )
            else:
                rows.append(self._html_progress_row("Qwen pages: waiting", 0, None))
        rows.append(
            self._html_progress_row(
                self._description,
                self._files_completed,
                self._total_files,
            )
        )
        return (
            '<div style="font-family:var(--vscode-editor-font-family,monospace);'
            'font-size:var(--vscode-editor-font-size,13px);padding:0.25rem 0">'
            + "".join(rows)
            + "</div>"
        )

    def _refresh_notebook(self) -> None:
        if self._html_factory is None or self._display is None:
            return
        content = self._html_factory(self._notebook_html())
        if self._display_handle is None:
            self._display_handle = self._display(content, display_id=True)
        else:
            self._display_handle.update(content)

    def close(self) -> None:
        if self._notebook_mode:
            self._refresh_notebook()
            return
        if self._page_bar is not None:
            self._page_bar.close()
        if self._file_bar is not None:
            self._file_bar.close()

In [ ]:
# | export
async def ocr_folder(
    root_folder: Path | str,
    *,
    base_url: str | None = None,
    model: str | None = None,
    output_dir_name: str = ".md_dashscope",
    prompt: str = _DEFAULT_PROMPT,
    dpi: int = 200,
    overwrite: bool = False,
    request_timeout_s: float = 180,
    max_concurrency: int | None = None,
    page_concurrency: int | None = None,
    high_resolution_images: bool = True,
    show_page_progress: bool = True,
    client: AsyncOpenAI | None = None,
) -> list[DashScopeOCRResult]:
    """Concurrently parse PDFs and images below a folder with DashScope Qwen."""
    root = _resolve_root(root_folder)
    selected_output_dir = _resolve_output_dir_name(output_dir_name)
    selected_model = _resolve_model(model)
    selected_prompt = _resolve_prompt(prompt)
    if dpi <= 0:
        raise ValueError("dpi must be greater than zero")
    if request_timeout_s <= 0:
        raise ValueError("request_timeout_s must be greater than zero")
    selected_max_concurrency = _positive_int(
        max_concurrency,
        "OPENAILIKED_MAX_CONCURRENCY",
        8,
    )
    selected_page_concurrency = _positive_int(
        page_concurrency,
        "OPENAILIKED_PAGE_CONCURRENCY",
        2,
    )

    jobs = _ocr_jobs(root, selected_output_dir)
    if not jobs:
        print(f"No PDF, PNG, or JPEG files found under {root}")
        return []

    results_by_index: dict[int, DashScopeOCRResult] = {}
    pending_jobs: list[tuple[int, Path, Path, Path]] = []
    for index, (source_path, markdown_path, metadata_path) in enumerate(jobs):
        if not overwrite and markdown_path.is_file():
            results_by_index[index] = DashScopeOCRResult(
                source_path,
                markdown_path,
                metadata_path,
                "skipped",
            )
        else:
            pending_jobs.append(
                (index, source_path, markdown_path, metadata_path)
            )

    if not pending_jobs:
        results = [results_by_index[index] for index in range(len(jobs))]
        print(f"Qwen parsing complete: 0 processed, {len(results)} skipped, 0 failed")
        return results

    parser_client = (
        client
        if client is not None
        else _create_client(base_url, request_timeout_s)
    )
    description = (
        "Qwen files (DashScope, "
        f"files/requests={selected_max_concurrency}, "
        f"pages/PDF={selected_page_concurrency})"
    )
    progress = _OCRFolderProgress(
        len(pending_jobs),
        description,
        show_page_progress,
    )
    file_semaphore = asyncio.Semaphore(selected_max_concurrency)
    request_semaphore = asyncio.Semaphore(selected_max_concurrency)

    async def run_job(
        index: int,
        source_path: Path,
        markdown_path: Path,
        metadata_path: Path,
    ) -> tuple[int, DashScopeOCRResult]:
        async with file_semaphore:
            is_pdf = source_path.suffix.casefold() == ".pdf"
            progress.start_file(index, source_path, None if is_pdf else 1)

            def report_pages_total(pages_total: int) -> None:
                progress.set_pages_total(index, pages_total)

            def report_page(
                pages_completed: int,
                pages_total: int,
                elapsed_s: float,
            ) -> None:
                progress.complete_page(
                    index,
                    pages_completed,
                    pages_total,
                    elapsed_s,
                )

            try:
                if is_pdf:
                    result = await ocr_pdf(
                        source_path,
                        markdown_path,
                        client=parser_client,
                        model=selected_model,
                        prompt=selected_prompt,
                        dpi=dpi,
                        overwrite=overwrite,
                        high_resolution_images=high_resolution_images,
                        page_concurrency=selected_page_concurrency,
                        request_semaphore=request_semaphore,
                        page_started=(
                            report_pages_total if show_page_progress else None
                        ),
                        page_progress=(report_page if show_page_progress else None),
                    )
                else:
                    result = await ocr_image(
                        source_path,
                        markdown_path,
                        client=parser_client,
                        model=selected_model,
                        prompt=selected_prompt,
                        overwrite=overwrite,
                        high_resolution_images=high_resolution_images,
                        request_semaphore=request_semaphore,
                    )
                    if result.status == "processed" and show_page_progress:
                        progress.complete_page(index, 1, 1, result.elapsed_s)
            except Exception as error:
                result = DashScopeOCRResult(
                    source_path,
                    markdown_path,
                    metadata_path,
                    "failed",
                    error=f"{type(error).__name__}: {error}",
                )
            finally:
                progress.finish_file(index)
            return index, result

    tasks = [
        asyncio.create_task(run_job(index, source, markdown, metadata))
        for index, source, markdown, metadata in pending_jobs
    ]
    completion_records: list[str] = []
    try:
        for completed_task in asyncio.as_completed(tasks):
            index, result = await completed_task
            results_by_index[index] = result
            error_text = f" | error={result.error}" if result.error else ""
            completion_records.append(
                f"Qwen task: {result.source_path} | status={result.status} | "
                f"elapsed={result.elapsed_s:.2f}s{error_text}"
            )
    finally:
        for task in tasks:
            if not task.done():
                task.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)
        progress.close()
        if client is None:
            await parser_client.close()

    for record in reversed(completion_records):
        print(record)
    results = [results_by_index[index] for index in range(len(jobs))]
    counts = Counter(result.status for result in results)
    print(
        "Qwen parsing complete: "
        f"{counts['processed']} processed, "
        f"{counts['skipped']} skipped, "
        f"{counts['failed']} failed"
    )
    return results

## Configuration and batch execution

Set `PDF_ROOT` to a folder containing PDFs and/or images. The call is intentionally opt-in because it consumes DashScope quota. Existing Markdown targets are skipped unless `OVERWRITE` is enabled. The default output tree is separate from local OCR output, making side-by-side comparison straightforward.

In [ ]:
from pathlib import Path

PDF_ROOT = PROJ_ROOT / "res" / "PDF-20260721"
OUTPUT_DIR_NAME = ".md_dashscope"
OCR_MODEL: str | None = None
OCR_DPI = 200
REQUEST_TIMEOUT_S = 180
MAX_CONCURRENCY = int(os.getenv("OPENAILIKED_MAX_CONCURRENCY", "8"))
PAGE_CONCURRENCY = int(os.getenv("OPENAILIKED_PAGE_CONCURRENCY", "2"))
HIGH_RESOLUTION_IMAGES = True
OVERWRITE = False
SHOW_PAGE_PROGRESS = True

In [ ]:
# | notest
# Uncomment or run this cell explicitly to consume DashScope quota.
# results = await ocr_folder(
#     PDF_ROOT,
#     model=OCR_MODEL,
#     output_dir_name=OUTPUT_DIR_NAME,
#     dpi=OCR_DPI,
#     overwrite=OVERWRITE,
#     request_timeout_s=REQUEST_TIMEOUT_S,
#     max_concurrency=MAX_CONCURRENCY,
#     page_concurrency=PAGE_CONCURRENCY,
#     high_resolution_images=HIGH_RESOLUTION_IMAGES,
#     show_page_progress=SHOW_PAGE_PROGRESS,
# )

In [ ]:
# | hide
from contextlib import redirect_stdout
from io import StringIO
from types import SimpleNamespace
from unittest.mock import patch

from fastcore.test import test_eq


class FakeAsyncOpenAIClient:
    def __init__(self, responses=(), delay_s=0.001):
        self.responses = list(responses)
        self.delay_s = delay_s
        self.calls = []
        self.active_calls = 0
        self.max_active_calls = 0
        self.closed = False
        self.chat = SimpleNamespace(
            completions=SimpleNamespace(create=self._create)
        )

    async def _create(self, **kwargs):
        self.calls.append(kwargs)
        if not self.responses:
            raise AssertionError("Unexpected chat completion call")
        response = self.responses.pop(0)
        delay_s = self.delay_s
        if isinstance(response, tuple):
            delay_s, response = response
        self.active_calls += 1
        self.max_active_calls = max(self.max_active_calls, self.active_calls)
        try:
            await asyncio.sleep(delay_s)
        finally:
            self.active_calls -= 1
        if isinstance(response, Exception):
            raise response
        if not isinstance(response, str):
            return response
        call_number = len(self.calls)
        return SimpleNamespace(
            id=f"response-{call_number}",
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content=response),
                    finish_reason="stop",
                )
            ],
            usage=SimpleNamespace(
                prompt_tokens=100 + call_number,
                completion_tokens=10 + call_number,
                total_tokens=110 + 2 * call_number,
            ),
        )

    async def close(self):
        self.closed = True


async def assert_async_fails(awaitable, contains: str):
    try:
        await awaitable
    except Exception as error:
        assert contains.casefold() in str(error).casefold()
    else:
        raise AssertionError("Expected awaitable to fail")


def make_pdf(path: Path, labels=("page",), password: str | None = None):
    document = pymupdf.open()
    for label in labels:
        page = document.new_page(width=240, height=120)
        page.insert_text((24, 60), label)
    if password is None:
        document.save(path)
    else:
        document.save(
            path,
            encryption=pymupdf.PDF_ENCRYPT_AES_256,
            owner_pw="owner-password",
            user_pw=password,
        )
    document.close()


def make_image(path: Path, label: str = "image"):
    document = pymupdf.open()
    page = document.new_page(width=240, height=120)
    page.insert_text((24, 60), label)
    page.get_pixmap(alpha=False).save(path)
    document.close()

In [ ]:
# | hide
def test_discovery_and_mapping():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory).resolve()
        (root / "nested").mkdir()
        (root / ".md_dashscope").mkdir()
        make_pdf(root / "B.PDF")
        make_image(root / "nested" / "a.JPG")
        make_image(root / "nested" / "C.PNG")
        make_image(root / ".md_dashscope" / "ignored.png")
        jobs = _ocr_jobs(root, ".md_dashscope")
        test_eq(
            [source.relative_to(root).as_posix() for source, _, _ in jobs],
            ["B.PDF", "nested/a.JPG", "nested/C.PNG"],
        )
        test_eq(
            [markdown.relative_to(root).as_posix() for _, markdown, _ in jobs],
            [
                ".md_dashscope/B.md",
                ".md_dashscope/nested/a.md",
                ".md_dashscope/nested/C.md",
            ],
        )
        test_eq(
            [metadata.name for _, _, metadata in jobs],
            ["B.qwen.json", "a.qwen.json", "C.qwen.json"],
        )

        make_pdf(root / "same.pdf")
        make_image(root / "same.jpg")
        try:
            _ocr_jobs(root, ".md_dashscope")
        except ValueError as error:
            assert "collision" in str(error).casefold()
        else:
            raise AssertionError("Expected an output collision")

        for invalid in ("", ".", "..", "a/b", str(root / "absolute")):
            try:
                _resolve_output_dir_name(invalid)
            except ValueError:
                pass
            else:
                raise AssertionError(f"Expected invalid output directory: {invalid}")


test_discovery_and_mapping()

In [ ]:
# | hide
async def test_pdf_payload_order_and_sidecar():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "two-pages.pdf"
        target = root / "two-pages.md"
        make_pdf(source, ("one", "two"))
        first = '<h1 data-bbox="0 0 999 100">One</h1>'
        second = "![figure](position:100,100,900,900)"
        client = FakeAsyncOpenAIClient(((0.03, first), (0.005, second)))
        result = await ocr_pdf(
            source,
            target,
            client=client,
            page_concurrency=2,
        )
        assert result.status == "processed", result.error
        test_eq(result.pages_total, 2)
        test_eq(result.pages_completed, 2)
        assert client.max_active_calls == 2
        test_eq(
            target.read_text(encoding="utf-8"),
            f"<!-- Page 1 -->\n\n{first}\n\n<!-- Page 2 -->\n\n{second}\n",
        )

        for call in client.calls:
            test_eq(call["model"], "qwen3.7-plus")
            test_eq(call["temperature"], 0)
            test_eq(
                call["extra_body"],
                {
                    "enable_thinking": False,
                    "vl_high_resolution_images": True,
                },
            )
            content = call["messages"][0]["content"]
            test_eq(content[1], {"type": "text", "text": "qwenvl markdown"})
            image_url = content[0]["image_url"]["url"]
            assert image_url.startswith("data:image/png;base64,")
            assert base64.b64decode(image_url.split(",", 1)[1]).startswith(b"\x89PNG")

        metadata = json.loads(
            target.with_suffix(".qwen.json").read_text(encoding="utf-8")
        )
        test_eq(metadata["prompt"], "qwenvl markdown")
        test_eq(metadata["model"], "qwen3.7-plus")
        test_eq([page["content"] for page in metadata["pages"]], [first, second])
        test_eq([page["page_number"] for page in metadata["pages"]], [1, 2])
        assert metadata["pages"][0]["usage"]["prompt_tokens"] > 0
        assert metadata["pages"][0]["response_id"].startswith("response-")
        assert metadata["pages"][0]["width"] > 0
        assert metadata["pages"][0]["height"] > 0


await test_pdf_payload_order_and_sidecar()

In [ ]:
# | hide
async def test_image_preservation_skip_and_failed_overwrite():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        source = root / "图像.png"
        target = root / "结果.md"
        make_image(source)
        raw = "```markdown\n<table data-bbox=\"0 0 999 999\">表格</table>\n```"
        first_client = FakeAsyncOpenAIClient((raw,))
        first = await ocr_image(source, target, client=first_client)
        assert first.status == "processed", first.error
        assert raw in target.read_text(encoding="utf-8")
        old_markdown = target.read_bytes()
        old_metadata = target.with_suffix(".qwen.json").read_bytes()

        skipped_client = FakeAsyncOpenAIClient()
        skipped = await ocr_image(source, target, client=skipped_client)
        test_eq(skipped.status, "skipped")
        test_eq(len(skipped_client.calls), 0)

        failed = await ocr_image(
            source,
            target,
            client=FakeAsyncOpenAIClient((RuntimeError("cloud failure"),)),
            overwrite=True,
        )
        test_eq(failed.status, "failed")
        test_eq(target.read_bytes(), old_markdown)
        test_eq(target.with_suffix(".qwen.json").read_bytes(), old_metadata)


def test_atomic_pair_rolls_back_publish_failure():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        target = root / "document.md"
        metadata_path = target.with_suffix(".qwen.json")
        target.write_text("old markdown", encoding="utf-8")
        metadata_path.write_text('{"old": true}', encoding="utf-8")
        original_replace = Path.replace

        def fail_markdown_publish(path, destination):
            destination = Path(destination)
            if destination == target and Path(path).name == target.name:
                raise OSError("simulated publish failure")
            return original_replace(path, destination)

        with patch.object(Path, "replace", fail_markdown_publish):
            try:
                _publish_output_pair(target, "new markdown", {"new": True})
            except OSError as error:
                assert "simulated" in str(error)
            else:
                raise AssertionError("Expected publish failure")
        test_eq(target.read_text(encoding="utf-8"), "old markdown")
        test_eq(
            metadata_path.read_text(encoding="utf-8"),
            '{"old": true}',
        )


await test_image_preservation_skip_and_failed_overwrite()
test_atomic_pair_rolls_back_publish_failure()

In [ ]:
# | hide
def test_jpeg_fallback_and_oversized_failure():
    with TemporaryDirectory() as temporary_directory:
        path = Path(temporary_directory) / "noise.png"
        Image.effect_noise((900, 900), 100).convert("RGB").save(path)
        pixmap = _load_image_pixmap(path)
        png_url = _data_url(pixmap.tobytes("png"), "image/png")
        jpeg_url = _data_url(
            pixmap.tobytes("jpeg", jpg_quality=90),
            "image/jpeg",
        )
        assert len(jpeg_url) < len(png_url)
        with patch.dict(globals(), {"_MAX_DATA_URL_BYTES": len(jpeg_url)}):
            selected_url, mime_type = _image_data_url(pixmap)
            test_eq(mime_type, "image/jpeg")
            assert selected_url.startswith("data:image/jpeg;base64,")
        with patch.dict(globals(), {"_MAX_DATA_URL_BYTES": len(jpeg_url) - 1}):
            try:
                _image_data_url(pixmap)
            except ValueError as error:
                assert "10 MiB" in str(error)
            else:
                raise AssertionError("Expected oversized payload failure")


test_jpeg_fallback_and_oversized_failure()

In [ ]:
# | hide
async def test_invalid_documents_and_responses():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        corrupt = root / "corrupt.pdf"
        corrupt.write_bytes(b"not a PDF")
        corrupt_result = await ocr_pdf(
            corrupt,
            root / "corrupt.md",
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(corrupt_result.status, "failed")
        assert not (root / "corrupt.md").exists()

        encrypted = root / "encrypted.pdf"
        make_pdf(encrypted, password="secret")
        encrypted_result = await ocr_pdf(
            encrypted,
            root / "encrypted.md",
            client=FakeAsyncOpenAIClient(),
        )
        test_eq(encrypted_result.status, "failed")
        assert "password" in encrypted_result.error.casefold()

        image = root / "empty.png"
        make_image(image)
        empty_response = SimpleNamespace(
            choices=[
                SimpleNamespace(
                    message=SimpleNamespace(content="  "),
                    finish_reason="stop",
                )
            ]
        )
        empty_result = await ocr_image(
            image,
            root / "empty.md",
            client=FakeAsyncOpenAIClient((empty_response,)),
        )
        test_eq(empty_result.status, "failed")
        assert not (root / "empty.md").exists()

        malformed_result = await ocr_image(
            image,
            root / "malformed.md",
            client=FakeAsyncOpenAIClient((SimpleNamespace(choices=[]),)),
        )
        test_eq(malformed_result.status, "failed")


await test_invalid_documents_and_responses()

In [ ]:
# | hide
async def test_folder_concurrency_continuation_and_output():
    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        for name in ("a.png", "b.png", "c.png", "skipped.png"):
            make_image(root / name, name)
        output_root = root / ".md_dashscope"
        output_root.mkdir()
        (output_root / "skipped.md").write_text("existing", encoding="utf-8")
        client = FakeAsyncOpenAIClient(
            ((0.02, "A"), RuntimeError("B failed"), (0.02, "C"))
        )
        output = StringIO()
        with redirect_stdout(output):
            results = await ocr_folder(
                root,
                client=client,
                max_concurrency=2,
                page_concurrency=2,
                show_page_progress=False,
            )
        test_eq(
            [result.status for result in results],
            ["processed", "failed", "processed", "skipped"],
        )
        assert 1 < client.max_active_calls <= 2
        test_eq(len(client.calls), 3)
        assert (output_root / "a.md").is_file()
        assert not (output_root / "b.md").exists()
        assert (output_root / "c.qwen.json").is_file()
        printed = output.getvalue()
        assert str(root / "a.png") in printed
        assert str(root / "b.png") in printed
        assert str(root / "c.png") in printed
        assert str(root / "skipped.png") not in printed
        assert "1 skipped" in printed
        assert "elapsed=" in printed
        assert not client.closed


def test_nested_progress_html():
    progress = object.__new__(_OCRFolderProgress)
    progress._show_pages = True
    progress._active_order = [1, 2]
    progress._active = {
        1: _ActivePageProgress(Path("old.pdf"), 5, 3, 1.0),
        2: _ActivePageProgress(Path("new.pdf"), 4, 2, 0.5),
    }
    progress._description = "Qwen files"
    progress._files_completed = 3
    progress._total_files = 7
    html = progress._notebook_html()
    assert html.index("new.pdf") < html.index("Qwen files")
    assert "old.pdf" not in html
    assert "2/4" in html
    assert "3/7" in html


await test_folder_concurrency_continuation_and_output()
test_nested_progress_html()

In [ ]:
# | hide
async def test_configuration_and_client_validation():
    with patch.dict(os.environ, {}, clear=False):
        os.environ.pop("OPENAILIKED_OCR_MODEL", None)
        test_eq(_resolve_model(None), "qwen3.7-plus")
        os.environ["OPENAILIKED_OCR_MODEL"] = "environment-model"
        test_eq(_resolve_model(None), "environment-model")
        test_eq(_resolve_model("explicit-model"), "explicit-model")

    with TemporaryDirectory() as temporary_directory:
        root = Path(temporary_directory)
        make_image(root / "input.png")
        for kwargs, expected in (
            ({"dpi": 0}, "dpi"),
            ({"request_timeout_s": 0}, "request_timeout"),
            ({"max_concurrency": 0}, "max_concurrency"),
            ({"page_concurrency": 0}, "page_concurrency"),
            ({"output_dir_name": "a/b"}, "output_dir_name"),
            ({"prompt": " "}, "prompt"),
            ({"model": " "}, "model"),
        ):
            await assert_async_fails(
                ocr_folder(root, client=FakeAsyncOpenAIClient(), **kwargs),
                expected,
            )

        with patch.dict(os.environ, {"DASHSCOPE_API_KEY": ""}, clear=False):
            await assert_async_fails(
                ocr_folder(root),
                "DASHSCOPE_API_KEY",
            )
        with patch.dict(
            os.environ,
            {"DASHSCOPE_API_KEY": "secret", "DASHSCOPE_API_URL": "invalid"},
            clear=False,
        ):
            await assert_async_fails(
                ocr_folder(root),
                "OpenAI-compatible",
            )

    await assert_async_fails(
        ocr_folder(Path("definitely-missing-folder")),
        "does not exist",
    )


await test_configuration_and_client_validation()

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()